# Exploring Free Energy Protocol settings

The free energy protocols implemented in **openfe** expose a number of simulation settings that can affect the convergence properties, accuracy and precision of an alchemical transformation. In this notebook, we will explore changes to common settings, advising when you might want to consider changing them and how to change them in practice. For a detailed explanation of the individual protocols and their limitations see the [protocol documentation](https://docs.openfree.energy/en/latest/guide/protocols/) and references therein.

The default settings for a `Protocol` can be inspected via the common `default_settings()` class method. The following example shows how to retrieve the default settings for the `RelativeHybridTopologyProtocol`:

In [1]:
from openfe.protocols.openmm_rfe import RelativeHybridTopologyProtocol

settings = RelativeHybridTopologyProtocol.default_settings()

## Sampling and performance settings

This section covers the settings that affect sampling, computational cost and statistical precision.

### Inspecting the default settings

The default settings for each of the protocols are designed to be a starting point for most alchemical transformations with the goal of obtaining reliable estimates with sufficient precision for decision-making at a reasonable computational cost. However, depending on the specific system and transformation being studied, you may want to adjust these settings to improve convergence or precision of the estimation.

<div class="alert alert-info">

**Tip:** For a complete explanation of each setting and how to customize the object see <a href="https://docs.openfree.energy/en/latest/cookbook/choose_protocol.html">Choose and Configure a Protocol</a>.

</div>


The main parameters that determine the amount of sampling performed and the associated setting for the `RelativeHybridTopologyProtocol` are:

| Setting                         | Description                                                                                                              | Setting           | Type |
|---------------------------------|--------------------------------------------------------------------------------------------------------------------------|--------------------------------------------|----------|
| **Lambda windows**              | The number of discrete alchemical states simulated during the transformation.                                            | `lambda_settings.lambda_windows`           | `int`    |
| **Production length**           | The length of the production simulation performed at each lambda window.                                                 | `simulation_settings.production_length`    |  `openff.units.unit.Quantity` |
| **Equilibration length**        | The length of the equilibration simulation performed at each lambda window using the selected multisate sampling method. | `simulation_settings.equilibration_length` |            `openff.units.unit.Quantity`              |
| **Protocol repeats**            | The number of independent executions of the protocol.                                                                    | `protocol_repeats`                         |  `int`    |

The appropriate changes to these settings are transformation dependent with no single setting guaranteeing sufficient convergence or accuracy in general. These settings provide a good starting point to investigate the affect of additional sampling. The following example retrieves them from the default settings:

In [2]:
n_repeats = settings.protocol_repeats
n_windows = settings.lambda_settings.lambda_windows
production_length = settings.simulation_settings.production_length
equilibration_length = settings.simulation_settings.equilibration_length

print(f"Lambda windows: {n_windows}")
print(f"Production length per state: {production_length}")
print(f"Equilibration length per state: {equilibration_length}")
print(f"Independent repeats: {n_repeats}")

Lambda windows: 11
Production length per state: 5.0 nanosecond
Equilibration length per state: 1.0 nanosecond
Independent repeats: 3


### Estimating the computational cost of a protocol

A useful first approximation of the computational cost of a protocol execution is the total alchemical simulation time in nanoseconds across all lambda windows and repeats. This can be calculated as:

$$T_{total} = N_{repeats} \times N_{\lambda windows} \times (T_{production} + T_{equilibration})$$

This formula applies to all multistate protocols including:

- `RelativeHybridTopologyProtocol`
- `AbsoluteBindingProtocol`
- `SepTopProtocol`
- `AbsoluteSolvationProtocol`

<div alert class="alert alert-info">

**Note:** This formula does not include the cost of an initial MD equilibration simulation that is used by some protocols, see the section on other protocols for more information.

</div>

So for the default settings the total simulation time is:

In [3]:
def estimate_rbfe_cost(settings):
    n_repeats = settings.protocol_repeats
    n_windows = settings.lambda_settings.lambda_windows
    production_length = settings.simulation_settings.production_length
    equilibration_length = settings.simulation_settings.equilibration_length

    total_sim_time = n_repeats * n_windows * (production_length + equilibration_length)
    return total_sim_time

total_sim_time_per_leg = estimate_rbfe_cost(settings)
print(f"Total simulation time: {total_sim_time_per_leg}")

Total simulation time: 198.0 nanosecond


The `RelativeHybridTopologyProtocol` simulates the complex and solvent legs of a transformation independently, so if both use the same settings the total cost of the protocol is double the above estimate. It is useful to consider this estimate when deciding how to adjust the settings for a given transformation.

### Increasing simulation length

This section describes how to increase the amount of production simulation performed at each lambda window and estimate the increase in computational cost.

<div class="alert alert-warning">

**Warning:** Extending an existing simulation is not currently supported. Increasing the simulation length after running with default settings means the protocol must be run again from the start with the new settings, so the full cost of the longer simulation is incurred.

</div>

#### When to increase simulation length

Consider increasing the simulation length **before** running the simulation when:

- The transformation involves large structural changes due to changes in formal charge, for example. In such cases, consider increasing the number of lambda windows; see the next section for more information.

Consider increasing the simulation length **after** running the simulation when:

- Free-energy estimates differ substantially between independent repeats of the protocol.
- Forward and reverse free-energy estimates disagree by more than 1kbT at 50% of the simulation length and their uncertainties do not overlap.[[1]](https://livecomsjournal.org/index.php/livecoms/article/view/v2i1e18378)
- A large number of samples are discarded during analysis due to the automated equilibration and decorrelation methods.
- Replica exchange or MBAR analysis indicates that alchemical overlap or replica mixing is poor between lambda windows. In such cases, consider increasing the number of lambda windows; see the next section for more information.

This information is available through the automated analysis performed by the protocols, see the list of available [analysis metrics](https://docs.openfe.org/en/latest/guide/results/simulation_health.html#userguide-simulation-health) for more information. It is recommended to review the analysis metrics before deciding to increase the simulation length.

#### When not to increase simulation length

Increasing the simulation length is not always the best solution. For example changes to the number of lambda windows or the spacing between them can sometimes be more effective when you have poor overlap between neighboring windows. In some cases changes to both simulation time and the lambda schedule maybe required.

#### Changing the production length

The following example shows how to double the production simulation length for the `RelativeHybridTopologyProtocol` and then calculates the new total simulation time and compares it to the default settings.


In [4]:
from openff.units import unit

long_settings = RelativeHybridTopologyProtocol.default_settings()

# Increase the production length to 10 ns
long_settings.simulation_settings.production_length = 10 * unit.nanoseconds

# Create a new protocol object with the updated settings
protocol = RelativeHybridTopologyProtocol(settings=long_settings)

print(f"Updated production length: {protocol.settings.simulation_settings.production_length}")

# Calculate the new total simulation time
new_total_sim_time_per_leg = estimate_rbfe_cost(protocol.settings)
print(f"New total simulation time: {new_total_sim_time_per_leg}")

# Print the percentage increase in total simulation time
percentage_increase = ((new_total_sim_time_per_leg - total_sim_time_per_leg) / total_sim_time_per_leg) * 100
print(f"Percentage increase in total simulation time: {percentage_increase.m:.2f}%")

Updated production length: 10 nanosecond
New total simulation time: 363.0 nanosecond
Percentage increase in total simulation time: 83.33%


### Increasing the number of lambda windows

Increasing the number of lambda windows introduces additional intermediate alchemical states along the transformation which can make the pathway easier to traverse and improve the overlap between neighboring states. This section describes how to increase the number of lambda windows and estimate the increase in computational cost.

<div class="alert alert-warning">

**Warning:** Running additional lambda windows for a complete simulation is not supported. Increasing the number of lambda windows after running with default settings means the protocol must be run again from the start with the new settings, so the full cost of the longer simulation is incurred.

</div>

#### When to increase the number of lambda windows

Consider increasing the number of lambda windows **before** running the simulation when:

- The transformation involves a change in net formal charge or substantial redistribution of charge, you may also need to consider increasing the simulation length see the previous section for more information.

Consider increasing the number of lambda windows **after** running the simulation when:

- Neighboring lambda windows have poor phase-space overlap or replica mixing statistics.

#### When not to increase the number of lambda windows

Increasing the number of lambda windows is not always the best solution, slow changes in ligand conformation or binding site rearrangements may require longer simulation lengths to achieve convergence.

#### Changing the number of lambda windows

The following example shows how to double the number of lambda windows for the `RelativeHybridTopologyProtocol` to `22` and calculates the new total simulation time. Note the `lambda_windows` setting only accepts integer values and generates a uniform spacing of lambda values between `0` and `1`.


In [5]:
more_window_settings = RelativeHybridTopologyProtocol.default_settings()

# Increase the number of lambda windows to 22
more_window_settings.lambda_settings.lambda_windows = 22
# The number of replicas must be increased to match the number of lambda windows
more_window_settings.simulation_settings.n_replicas = 22

# Create a new protocol object with the updated settings
protocol = RelativeHybridTopologyProtocol(settings=more_window_settings)

print(f"Updated number of lambda windows: {protocol.settings.lambda_settings.lambda_windows}")

# Calculate the new total simulation time
new_total_sim_time_per_leg = estimate_rbfe_cost(protocol.settings)
print(f"New total simulation time: {new_total_sim_time_per_leg}")

# Print the percentage increase in total simulation time
percentage_increase = ((new_total_sim_time_per_leg - total_sim_time_per_leg) / total_sim_time_per_leg) * 100
print(f"Percentage increase in total simulation time: {percentage_increase.m:.2f}%")

Updated number of lambda windows: 22
New total simulation time: 396.0 nanosecond
Percentage increase in total simulation time: 100.00%


### Increasing the number of independent repeats


Independent repeats run the same protocol from the same input structures but different initial velocities. By default, **openfe**  reports the **average** free energy change of the independent repeats as its aggregated estimate and the sample standard deviation is used to quantify the uncertainty of the prediction. Adding more repeats is a simple way to refine the estimate of the mean and uncertainty, but at a linear increase in computational cost. See the best practices paper by [May et al.](https://livecomsjournal.org/index.php/livecoms/article/view/v2i1e18378) for more information.

#### Changing the number of repeats

The following example shows how to increase the number of independent repeats per single execution of the `RelativeHybridTopologyProtocol` to `4`. Note that you can also execute the protocol multiple times with a single repeat per execution which may be more efficient, see the [quickrun guide](https://docs.openfree.energy/en/stable/guide/execution/quickrun_execution.html#parallel-execution-of-repeats-with-quickrun) for more information.

In [6]:
more_repeats_settings = RelativeHybridTopologyProtocol.default_settings()

# Increase the number of independent repeats to 4
more_repeats_settings.protocol_repeats = 4

# Create a new protocol object with the updated settings
protocol = RelativeHybridTopologyProtocol(settings=more_repeats_settings)

print(f"Updated number of independent repeats: {protocol.settings.protocol_repeats}")

# Calculate the new total simulation time
new_total_sim_time_per_leg = estimate_rbfe_cost(protocol.settings)
print(f"New total simulation time: {new_total_sim_time_per_leg}")

# Print the percentage increase in total simulation time
percentage_increase = ((new_total_sim_time_per_leg - total_sim_time_per_leg) / total_sim_time_per_leg) * 100
print(f"Percentage increase in total simulation time: {percentage_increase.m:.2f}%")

Updated number of independent repeats: 4
New total simulation time: 264.0 nanosecond
Percentage increase in total simulation time: 33.33%


### Increasing the equilibration length

The equilibration simulation uses the selected multisate sampling method to relax the system at each lambda window before production sampling begins. No samples are collected during this phase for the final free energy estimate.

#### When to increase the equilibration length

Consider increasing the equilibration length **before** running the simulation when:

- The input structures are far from equilibrium and the system would benefit from additional relaxation before production sampling begins.

#### When not to increase the equilibration length

If the system is expected to require a long equilibration time, such those with membrane bound proteins, it may be more efficient to perform a longer equilibration simulation outside the protocol.

#### Changing the equilibration length

The following example shows how to double the equilibration simulation length for the `RelativeHybridTopologyProtocol` and calculates the new total simulation time. Note that the equilibration length is set per lambda window, so increasing it will increase the total simulation time linearly with the number of windows see increasing production length for more information on the associated cost.

In [7]:
more_equilibration_settings = RelativeHybridTopologyProtocol.default_settings()

# Increase the equilibration length to 2 ns
more_equilibration_settings.simulation_settings.equilibration_length = 2 * unit.nanoseconds

# Create a new protocol object with the updated settings
protocol = RelativeHybridTopologyProtocol(settings=more_equilibration_settings)

print(f"Updated equilibration length: {protocol.settings.simulation_settings.equilibration_length}")

# Calculate the new total simulation time
new_total_sim_time_per_leg = estimate_rbfe_cost(protocol.settings)
print(f"New total simulation time: {new_total_sim_time_per_leg}")

# Print the percentage increase in total simulation time
percentage_increase = ((new_total_sim_time_per_leg - total_sim_time_per_leg) / total_sim_time_per_leg) * 100
print(f"Percentage increase in total simulation time: {percentage_increase.m:.2f}%")

Updated equilibration length: 2 nanosecond
New total simulation time: 231.0 nanosecond
Percentage increase in total simulation time: 16.67%


### Other protocols

The above concepts apply to most free energy protocols, but the settings are located differently. Here we will briefly cover the differences in setting layout for the following protocols:

- `AbsoluteBindingProtocol`
- `SepTopProtocol`

These protocols execute both the `complex` and `solvent` legs within a single transformation, the settings to control the production sampling are then prefixed with the leg name `complex_*` or `solvent_*`.

In addition, each leg has a `*_equil_simulation_settings` object controlling an initial MD simulation used to determine the restraint atoms in the complex and to seed both legs production simulations.


In [8]:
import warnings

# Silence a third-party Biopython deprecation warning raised on import
warnings.filterwarnings("ignore", message="The Bio.Application modules")

In [9]:
from openfe.protocols.openmm_afe import AbsoluteBindingProtocol
from openfe.protocols.openmm_septop import SepTopProtocol


#### Initial MD equilibration settings

Both protocols expose the initial MD settings for each leg as `complex_equil_simulation_settings` and `solvent_equil_simulation_settings`. These control the initial MD simulation (including NVT equilibration), not equilibration at each lambda window. The same setting names and approach apply to both protocols; here we update the NVT equilibration for both legs of an `AbsoluteBindingProtocol`.


In [10]:
abfe_settings = AbsoluteBindingProtocol.default_settings()

for leg in ("complex", "solvent"):
    equil_settings = getattr(abfe_settings, f"{leg}_equil_simulation_settings")
    equil_settings.equilibration_length_nvt = 2 * unit.nanoseconds

abfe_protocol = AbsoluteBindingProtocol(settings=abfe_settings)
print(f"Updated complex initial NVT equilibration: {abfe_protocol.settings.complex_equil_simulation_settings.equilibration_length_nvt}")
print(f"Updated solvent initial NVT equilibration: {abfe_protocol.settings.solvent_equil_simulation_settings.equilibration_length_nvt}")


Updated complex initial NVT equilibration: 2 nanosecond
Updated solvent initial NVT equilibration: 2 nanosecond


#### Estimating computational cost

For both protocols, each leg runs an initial MD simulation once per repeat, followed by equilibration and production at each lambda window. The same estimate applies to both:

$$T_{total} = N_{repeats} \times \sum_{legs} \left[ T_{MD} + N_{\lambda windows} \times (T_{production} + T_{equilibration}) \right]$$

Here, $T_{MD}$ is the total initial MD time set by `*_equil_simulation_settings`; the per-window lengths and window counts come from `*_simulation_settings` and the lambda schedule.


In [11]:
def estimate_abfe_septop_cost(settings):
    total_sim_time = 0 * unit.nanoseconds
    for leg in ("complex", "solvent"):
        equil = getattr(settings, f"{leg}_equil_simulation_settings")
        sampling = getattr(settings, f"{leg}_simulation_settings")

        md_time = equil.equilibration_length_nvt + equil.equilibration_length + equil.production_length
        replica_exchange_time = sampling.n_replicas * (sampling.production_length + sampling.equilibration_length)
        total_sim_time += md_time + replica_exchange_time

    return total_sim_time * settings.protocol_repeats

abfe_default_settings = AbsoluteBindingProtocol.default_settings()
septop_default_settings = SepTopProtocol.default_settings()
print(f"ABFE default total simulation time: {estimate_abfe_septop_cost(abfe_default_settings).m:.2f} ns")
print(f"SepTop default total simulation time: {estimate_abfe_septop_cost(septop_default_settings).m:.2f} ns")


ABFE default total simulation time: 1471.65 ns
SepTop default total simulation time: 1531.20 ns


### AbsoluteBindingProtocol - Increasing the number of lambda windows


ABFE uses one list of lambda values per scaled interaction (`lambda_elec`, `lambda_vdw` and `lambda_restraints`) allowing for exact control over the lambda spacing as well as number of windows. This requires you to define one value per window, with all lists for a leg having the same length, and that length must match `n_replicas` for the leg.


In [12]:
abfe_window_settings = AbsoluteBindingProtocol.default_settings()

# Define the new lambda schedule for the complex leg by adding 3 more charge windows
abfe_window_settings.complex_lambda_settings.lambda_elec = [0.0] * 6 + [0.1, 0.2, 0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.8, 0.85, 0.9] + [1.0] * 15
# Extra vdW windows are added to match the length of the new charge schedule
abfe_window_settings.complex_lambda_settings.lambda_vdw = [0.0] * 19 + [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
# Extra restraint windows are added to match the length of the new charge schedule
abfe_window_settings.complex_lambda_settings.lambda_restraints = [0.0, 0.2, 0.4, 0.6, 0.8] + [1.0] * 28

# The number of replicas must match the number of lambda windows
abfe_window_settings.complex_simulation_settings.n_replicas = len(abfe_window_settings.complex_lambda_settings.lambda_vdw)

# Create a new protocol object with the updated settings
abfe_protocol = AbsoluteBindingProtocol(settings=abfe_window_settings)

<div class="alert alert-info">

**Tip:** Changing the number of lambda windows or the lambda schedule for the **AbsoluteBindingProtocol** and **SepTopProtocol** by hand can be error-prone. The protocols provide a **validate** method to check that the lambda schedules are valid for all legs of the calculation, use them to validate your changes and catch issues before execution. This does require a representative system however, see the example below for more details.

</div>

In [13]:
# Validate the lambda schedules for the protocol, this requires a valid inputs for the transformation
from rdkit import Chem
import openfe

# Define a solvent for the calculation
solvent = openfe.SolventComponent()
# Load the protein and ligand from a pdb
protein = openfe.ProteinComponent.from_pdb_file("assets/tyk2_protein.pdb")
# Load the ligands from an SDF file
supplier = Chem.SDMolSupplier("assets/tyk2_ligands.sdf", removeHs=False)
ligands = [openfe.SmallMoleculeComponent.from_rdkit(mol) for mol in supplier if mol is not None]

# Build an example chemical system to validate the lambda schedule
system_a = openfe.ChemicalSystem({
    "ligand": ligands[0],
    "protein": protein,
    "solvent": solvent
})

system_b = openfe.ChemicalSystem({
    "protein": protein,
    "solvent": solvent
})

abfe_protocol = AbsoluteBindingProtocol(settings=abfe_window_settings)
abfe_protocol.validate(
    stateA=system_a,
    stateB=system_b,
    mapping=None,  # The mapping is not required for ABFE or SepTop, but is required for RBFE
)

### SepTopProtocol - Increasing the number of lambda windows

SepTop defines separate lambda lists for each scaled interaction and for each of the states A and B: `lambda_elec_A/B`, `lambda_vdw_A/B` and `lambda_restraints_A/B`. All lists for a leg must have the same length, and the matching `n_replicas` must equal the number of windows.


In [14]:
septop_window_settings = SepTopProtocol.default_settings()

# Define the new lambda schedule for the complex leg adding 3 more charge windows for A and B
septop_window_settings.complex_lambda_settings.lambda_elec_A = [0.0] * 8 + [0.25, 0.4, 0.5, 0.6, 0.75, 0.8] + [1.0] * 8
septop_window_settings.complex_lambda_settings.lambda_elec_B = [1.0] * 8 + [0.8, 0.75, 0.6, 0.5, 0.4, 0.25] + [0.0] * 8
septop_window_settings.complex_lambda_settings.lambda_vdw_A = [0.0] * 15 + [0.14, 0.28, 0.42, 0.57, 0.71, 0.85, 1.0]
septop_window_settings.complex_lambda_settings.lambda_vdw_B = [1.0, 0.85, 0.71, 0.57, 0.42, 0.28, 0.14] + [0.0] * 15
septop_window_settings.complex_lambda_settings.lambda_restraints_A = [0.0, 0.05, 0.1, 0.3, 0.5, 0.75] + [1.0] * 16
septop_window_settings.complex_lambda_settings.lambda_restraints_B = [1.0] * 16 + [0.75, 0.5, 0.3, 0.1, 0.05, 0.0]

# The number of replicas must match the number of lambda windows
septop_window_settings.complex_simulation_settings.n_replicas = len(septop_window_settings.complex_lambda_settings.lambda_vdw_A)

# Create a new protocol object with the updated settings
septop_protocol = SepTopProtocol(settings=septop_window_settings)


Validate the edited schedule before running the protocol using the `validate` method of the protocol, this requires a representative system to be provided.


In [15]:
# Validate the updated lambda schedule for the protocol using an example system with the inputs from the ABFE cell above
system_a = openfe.ChemicalSystem({
    "ligand": ligands[0],
    "protein": protein,
    "solvent": solvent
})
system_b = openfe.ChemicalSystem({
    "ligand": ligands[1],
    "protein": protein,
    "solvent": solvent
})

septop_protocol = SepTopProtocol(settings=septop_window_settings)
septop_protocol.validate(
    stateA=system_a,
    stateB=system_b,
    mapping=None,  # The mapping is not required for ABFE or SepTop, but is required for RBFE
)